# ML Security Basics

## What You Will Learn
- Understand common ML security threats: model poisoning, adversarial examples, and data leakage
- Apply encryption basics to protect ML artifacts at rest and in transit
- Implement secure configuration management practices
- Use Python cryptography libraries to hash and encrypt model files

## Why This Matters in MLOps
ML models are valuable intellectual property and attack vectors. In production MLOps pipelines, unencrypted model artifacts can be tampered with or stolen. Without integrity checks, a poisoned model could serve malicious predictions. Security must be built into every stage of the ML lifecycle — not bolted on at the end.

## You're Done When...
- You can explain three major ML security threats
- You have encrypted and decrypted a model artifact using symmetric encryption
- You have computed and verified file hashes for integrity checking
- You have completed the fill-in-the-blank exercises

---

### MLOps Perspective

In a typical MLOps workflow, models are trained on sensitive data, stored in registries, and served via APIs. Each handoff is a risk:
- **Training → Registry**: Model weights can be intercepted or replaced
- **Registry → Serving**: A compromised model can serve biased or harmful predictions
- **API → Client**: Predictions can leak training data attributes

By the end of this notebook you will apply cryptographic controls that address these risks.

---
## 1. Introduction to ML Security Threats

### Model Poisoning
Attackers inject malicious data during training to corrupt model behavior. A poisoned model may misclassify specific inputs or embed backdoors.

### Adversarial Examples
Small, imperceptible perturbations to input data cause the model to make incorrect predictions. These can bypass fraud detection or image classifiers.

### Data Leakage
Sensitive training data can be extracted from model outputs or gradients. Membership inference attacks determine if a record was in the training set.

**Defense layers**: Input validation, encryption, access controls, monitoring, and adversarial training.

---
## 2. File Hashing for Integrity Verification

Hashing creates a unique fingerprint of a file. Compare hashes before and after transfer to detect tampering.

In [ ]:
import hashlib
import os

# Create a sample model artifact (simulated)
sample_model_path = 'sample_model.pkl'
with open(sample_model_path, 'w') as f:
    f.write('model_weights_version_1:param1=0.45,param2=0.89')

# Compute SHA-256 hash
sha256_hash = hashlib.sha256()
with open(sample_model_path, 'rb') as f:
    for chunk in iter(lambda: f.read(4096), b''):
        sha256_hash.update(chunk)

print(f'SHA-256 hash: {sha256_hash.hexdigest()}')
print(f'File size: {os.path.getsize(sample_model_path)} bytes')

### Exercise: Verify File Integrity

Run the cell below to compute the hash. Then modify the file and re-run to see the hash change.

In [ ]:
# Store the original hash
original_hash = sha256_hash.hexdigest()
print(f'Original hash: {original_hash}')

# Simulate tampering
with open(sample_model_path, 'a') as f:
    f.write('\n# INJECTED_MALICIOUS_WEIGHT')

# Recompute hash
new_hash = hashlib.sha256()
with open(sample_model_path, 'rb') as f:
    for chunk in iter(lambda: f.read(4096), b''):
        new_hash.update(chunk)

print(f'New hash after tampering: {new_hash.hexdigest()}')
print(f'Integrity check passed: {original_hash == new_hash.hexdigest()}')

---
## 3. Encryption Basics for ML Artifacts

Encryption protects data **at rest** (on disk) and **in transit** (over a network). We use the `cryptography` library with Fernet symmetric encryption.

In [ ]:
from cryptography.fernet import Fernet

# Generate an encryption key
key = Fernet.generate_key()
cipher = Fernet(key)

print(f'Encryption key (keep this secret!): {key.decode()}')

# Encrypt the model artifact
with open(sample_model_path, 'rb') as f:
    model_data = f.read()

encrypted_data = cipher.encrypt(model_data)

# Save encrypted artifact
encrypted_path = 'sample_model.enc'
with open(encrypted_path, 'wb') as f:
    f.write(encrypted_data)

print(f'Encrypted model saved to {encrypted_path}')
print(f'Original size: {len(model_data)} bytes')
print(f'Encrypted size: {len(encrypted_data)} bytes')

In [ ]:
# Decrypt and verify
with open(encrypted_path, 'rb') as f:
    encrypted_data = f.read()

decrypted_data = cipher.decrypt(encrypted_data)

print('Decrypted content:')
print(decrypted_data.decode())
print(f'\nDecryption successful: {model_data == decrypted_data}')

---
## 4. Secure Configuration Management

Never hardcode secrets. Use environment variables or a secrets manager.

In [ ]:
import os

# Set environment variable (in production, use a .env file or secrets manager)
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'my-secure-token-here'

# Best practice: load from environment, never hardcode
tracking_password = os.getenv('MLFLOW_TRACKING_PASSWORD')

if tracking_password:
    print('Tracking password loaded securely from environment.')
else:
    print('WARNING: No tracking password set!')

---
## 5. Fill-in-the-Blank Exercises

Complete the code below by replacing `___` with the correct code.

In [ ]:
# Exercise 1: File hashing for integrity
import hashlib

def verify_file_integrity(filepath, expected_hash):
    """Compute SHA-256 hash of a file and compare to expected."""
    h = hashlib.____()  # Hint: use the SHA-256 algorithm
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b''):
            h.update(chunk)
    actual_hash = h.hexdigest()
    return actual_hash == expected_hash

# Test
test_hash = 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'
print(f'Integrity check: {verify_file_integrity(sample_model_path, test_hash)}')

In [ ]:
# Exercise 2: Encrypt a model artifact
from cryptography.fernet import Fernet

# Generate key and cipher
key = Fernet.____()  # Hint: generate a new key
cipher = Fernet(key)

# Read and encrypt model file
with open('sample_model.pkl', 'rb') as f:
    data = f.read()

encrypted = cipher.____(data)  # Hint: encrypt the data

print(f'Encrypted {len(data)} bytes -> {len(encrypted)} bytes')
print('Encryption complete!')

---
## Summary

- ML models face unique threats: poisoning, adversarial attacks, data leakage
- File hashing (SHA-256) provides integrity verification
- Symmetric encryption (Fernet) protects model artifacts at rest
- Never hardcode secrets — use environment variables or secrets managers

In the next notebook, you will apply governance and access control patterns to your MLflow model registry.